# Session 2: Make It Faster

This morning you trained on one GPU. Some of you tried 24,000 examples and waited. The
full complaint database has about 10 million complaints.

On this supercomputer, each node has exactly one GPU. To use more GPUs, you use more
nodes. Today your team runs **the same training job on 1, 2, and 4 nodes** and
measures how much faster it gets.

**Team roles** (pick now):

| Role | Job |
|---|---|
| Driver | Owns this session and runs the cells |
| Recorder | Fills in the prediction and results tables |
| Skeptic | Asks "why isn't it exactly 4 times faster?" and checks the numbers |
| Reporter | Presents your three numbers at Lightning Shares tomorrow morning |

## 0. Check your nodes

This session should have **4 nodes**. The second command asks each node for its GPU.

In [ ]:
import json, os, pathlib, subprocess
from IPython.display import Image, display

SHARED = "/work/10539/ashleyscruse/vista/inspire-shared"   # shared data, model, and Python
PY     = f"{SHARED}/venv/bin/python"                       # the workshop's Python
REPO   = str(pathlib.Path.cwd().parent)                    # this repository
RUNS   = f"{os.environ['SCRATCH']}/inspire"                # where your results go

os.environ.update(SHARED=SHARED, PY=PY)
assert pathlib.Path(PY).exists(), "Can't see the shared folder. Raise your hand."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

!scontrol show hostnames {os.environ['SLURM_JOB_NODELIST']}
!srun --overlap -N 4 -n 4 nvidia-smi --query-gpu=name --format=csv,noheader

## 1. How splitting the work works

Every node gets a full copy of the model. The training complaints are dealt out like
cards, so with 4 nodes each one reads a quarter of them. After every batch, the nodes
**compare notes**: they average what they learned and update their copies the same way,
so all four stay identical. This is called *data parallel* training.

Comparing notes takes time and travels over the network between nodes. Keep that in
mind when you predict.

## 2. Predict before you run

We'll train on **60,000 complaints**. Your 1-node time is the baseline. Fill in your
team's guesses:

In [ ]:
PREDICT_2_NODES = None   # seconds, e.g. 150
PREDICT_4_NODES = None   # seconds
WHY = "your reasoning in one sentence"

## 3. Run it on 1, 2, then 4 nodes

Each run trains on the same 60,000 complaints. Watch the step count: with more nodes,
each step handles more complaints at once, so there are fewer steps.

In [ ]:
N_EXAMPLES = 60000

!OUT={RUNS}/scale-1node bash {REPO}/scripts/scale.sh 1 --limit {N_EXAMPLES}

In [ ]:
!OUT={RUNS}/scale-2node bash {REPO}/scripts/scale.sh 2 --limit {N_EXAMPLES}

In [ ]:
!OUT={RUNS}/scale-4node bash {REPO}/scripts/scale.sh 4 --limit {N_EXAMPLES}

## 4. Results

**Speedup** = 1-node time ÷ this time. Perfect would be 2 on 2 nodes and 4 on 4.
**Efficiency** = speedup ÷ nodes. 100% means no time lost to comparing notes.

In [ ]:
runs = []
for n in (1, 2, 4):
    f = pathlib.Path(f"{RUNS}/scale-{n}node/results.json")
    if f.exists():
        runs.append(json.loads(f.read_text()))

base = runs[0]["train_seconds"]
print(f"{'nodes':>5} {'seconds':>8} {'speedup':>8} {'efficiency':>10} {'accuracy':>9}")
for r in runs:
    s = base / r["train_seconds"]
    print(f"{r['gpus']:>5} {r['train_seconds']:>8.0f} {s:>8.2f} {s / r['gpus']:>10.0%} {r['accuracy']:>9.0%}")

!{PY} {REPO}/scripts/plot_scaling.py {RUNS}/scale-1node {RUNS}/scale-2node {RUNS}/scale-4node {RUNS}/scaling.png
display(Image(f"{RUNS}/scaling.png"))

## 5. Talk it through

1. How close were your predictions? Which was further off?
2. Why isn't 4 nodes exactly 4 times faster? (Hint: what happens after every step?)
3. Did accuracy change? With 4 nodes, each step learns from 4 times as many complaints,
   but takes 4 times fewer steps. Is that the same training?
4. If you had 64 nodes, would it be 64 times faster? What would you need to check?

## 6. Stretch: push it

Pick one if you have time.

- **Bigger batches.** Change `--batch 32` to `--batch 64` on 4 nodes. Each node does
  more work between note-comparisons. Does efficiency go up? Does accuracy hold?
- **Bigger job.** Run all 120,000 complaints on 4 nodes. How long would 1 node have
  taken? (Estimate it from your table; don't run it.)

In [ ]:
!OUT={RUNS}/stretch bash {REPO}/scripts/scale.sh 4 --limit {N_EXAMPLES} --batch 64

## 7. For Lightning Shares tomorrow

Your reporter brings three numbers and one sentence:

- 1-node time: ______ seconds
- 4-node time: ______ seconds (speedup ______)
- Accuracy: ______
- What surprised us: ______________________________

**When you're done:** close this session so its nodes are freed.